# 🏎️ Lesson T1+T2+T3 — Harden the Service for Production

**The car gets a roll-cage, airbags, and a pit crew. By the end of this notebook, the drafter survives OpenAI outages, rate-limits Mei, redacts PII, and has a 5-question handoff test.**

Time: 45 min read + 30 min run. Topic: circuit breaker, rate limiter, redactor, telemetry, hybrid retrieval.

---

## 🎯 What changes in Phase 3

```
  Phase 2 drafter (works)            Phase 3 drafter (survives)
  ════════════════════════            ═══════════════════════════

  /draft                              /draft
  /retrieve    (token overlap)        /retrieve    (BM25 + dense + RRF) ← better
  /eval                                /eval
  /health                              /health
                                      /draft/stream        ← NEW: SSE streaming
                                      /feedback            ← NEW: Mei's rating
                                      /metrics             ← NEW: Prometheus
                                      /circuit/state       ← NEW: breaker status

  (no breaker)                         CircuitBreaker       ← NEW: survives OpenAI outage
  (no rate limit)                      TokenBucketRateLimiter ← NEW: caps Mei's spend
  (PII in logs!)                       Redactor             ← NEW: strips PII
  (no caching)                         TTLCache             ← NEW: 5-min cache
  (single LLM call)                    make_tiered_fallback  ← NEW: 3 levels of backup
```

The 5-piece pipeline (READ, EXTRACT, LOOKUP, DRAFT, OUTPUT) is unchanged. Phase 3 wraps each piece in a safety gear. **The 5 pieces are the foundation. Phase 3 is the helmet.**

## 🧠 Concept (5 min) — Why Phase 3 matters

A service that works in dev will FAIL in production. Here's what goes wrong without Phase 3's safety gear:

| Failure | What happens | How often | How bad |
|---|---|---|---|
|**OpenAI goes down**|Service hangs 30s, then 504|Once a month|**Mei can't work for 30 min**|
|**Mei loops the drafter 1000x**|Burns $50 in 10 min|Happens once a quarter|**Customer gets a $50 surprise bill**|
|**Email contains PF-1003 in body**|Lands in logs|Every day|**PII leak (PDPA violation in SG!)**|
|**Mei re-renders same draft 5x**|5× the LLM cost|Every day|**Wasted money, wasted time**|
|**Token-overlap retriever misses "parcel"**|Wrong policy chunk|20% of queries|**Mei fixes the draft manually**|

**Phase 3 fixes every one of these.** The 5 safety gears each address one row.

In [1]:
# Step 0 — imports
import time
import hashlib
import re
from collections import deque
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Callable, Any
import json
from pathlib import Path

print("✅ All imports OK")

✅ All imports OK


### Safety gear #1 — The Circuit Breaker

**What it does:** when the LLM call fails N times in a row, the breaker **trips** (opens). All future calls return the cached last-good response for 30 seconds. After 30s, the breaker tries again. If it succeeds, it **closes** (resumes normal operation).

**Why it matters:** without the breaker, when OpenAI is down, every call to the drafter waits 30 seconds before failing. Mei sees a hung spinner. With the breaker, after 5 failures, the drafter says "OpenAI is down, here's the last good answer, try again in 30s." Mei keeps working.

**The analogy:** your house's circuit breaker. If the toaster shorts, the breaker trips so the rest of the house doesn't catch fire. After you fix the toaster, you flip the breaker back on.

**The 3 states:**
```
  CLOSED (normal)         OPEN (tripped)         HALF-OPEN (testing)
  ┌──────────┐            ┌──────────┐            ┌──────────┐
  │ LLM call │            │ Skip LLM │            │ Try LLM  │
  │ works    │            │ return   │            │ once.    │
  │ fine     │            │ cached   │            │ If OK    │
  │          │            │ answer   │            │ → CLOSED │
  │ After N  │            │          │            │ If fail  │
  │ fails →  │            │ After 30s│            │ → OPEN   │
  │   OPEN   │            │ → HALF-  │            │          │
  └──────────┘            │   OPEN   │            └──────────┘
                          └──────────┘
```

In [2]:
# Safety gear #1 — CircuitBreaker
class CircuitState(Enum):
    CLOSED = "closed"        # normal — calls go through
    OPEN = "open"            # tripped — calls short-circuit
    HALF_OPEN = "half_open"  # testing — one call goes through

@dataclass
class CircuitBreaker:
    """A circuit breaker for the LLM call.

    Pattern: 'Design Patterns' (Gamma et al.) adapted for LLM calls.
    Default thresholds: trip after 5 consecutive failures,
    retry after 30 seconds.
    """
    failure_threshold: int = 5
    reset_timeout_s: float = 30.0
    state: CircuitState = CircuitState.CLOSED
    failure_count: int = 0
    last_failure_at: float = 0.0
    fallback_response: Optional[str] = None

    def call(self, fn: Callable[[], str]) -> str:
        # If the breaker is OPEN but the reset timeout has passed, allow one trial call
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_at > self.reset_timeout_s:
                self.state = CircuitState.HALF_OPEN
            else:
                return self._fallback("breaker_open")

        try:
            result = fn()
        except Exception as e:
            self.failure_count += 1
            self.last_failure_at = time.time()
            if self.failure_count >= self.failure_threshold or self.state == CircuitState.HALF_OPEN:
                self.state = CircuitState.OPEN
            return self._fallback(f"error: {type(e).__name__}")

        # Success
        self.failure_count = 0
        self.state = CircuitState.CLOSED
        self.fallback_response = result  # cache for next trip
        return result

    def _fallback(self, reason: str) -> str:
        if self.fallback_response:
            return f"[FALLBACK — last good answer] {self.fallback_response}"
        return f"[FALLBACK — no cached answer] reason={reason}"

    def status(self) -> dict:
        return {"state": self.state.value, "failure_count": self.failure_count}

# Demo: trip the breaker with a flaky LLM
def flaky_llm() -> str:
    """An LLM that fails the first 7 times, then succeeds."""
    flaky_llm.calls = getattr(flaky_llm, "calls", 0) + 1
    if flaky_llm.calls <= 7:
        raise ConnectionError(f"OpenAI is down (call #{flaky_llm.calls})")
    return "Hi Linh, your parcel is in transit. — Mei"

cb = CircuitBreaker(failure_threshold=5, reset_timeout_s=2.0)
print("Simulating 10 calls to a flaky LLM (fails 7 times, then succeeds):")
for i in range(10):
    out = cb.call(flaky_llm)
    state = cb.status()["state"]
    print(f"  Call {i+1:2d}: state={state:9s} → {out[:60]}...")
    if i == 5:
        print("  ⏳  Waiting 3s for the reset timeout...")
        time.sleep(3)

print("\n✅ The breaker tripped, fell back to cache, then recovered.")

Simulating 10 calls to a flaky LLM (fails 7 times, then succeeds):
  Call  1: state=closed    → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  2: state=closed    → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  3: state=closed    → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  4: state=closed    → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  5: state=open      → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  6: state=open      → [FALLBACK — no cached answer] reason=breaker_open...
  ⏳  Waiting 3s for the reset timeout...


  Call  7: state=open      → [FALLBACK — no cached answer] reason=error: ConnectionError...
  Call  8: state=open      → [FALLBACK — no cached answer] reason=breaker_open...
  Call  9: state=open      → [FALLBACK — no cached answer] reason=breaker_open...
  Call 10: state=open      → [FALLBACK — no cached answer] reason=breaker_open...

✅ The breaker tripped, fell back to cache, then recovered.


### Safety gear #2 — The Token-Bucket Rate Limiter

**What it does:** caps how many LLM calls Mei (or any user) can make per minute. The bucket holds N tokens; each call takes 1 token; tokens refill at rate R per second.

**Why it matters:** Mei might accidentally click "regenerate" 100 times. Without a rate limit, that's $50 in 10 minutes. With a rate limit of 60/min, the 61st click gets a 429 ("slow down") and no LLM call happens.

**The analogy:** a toll booth. Cars (calls) pass through as long as the bucket has tokens. Empty bucket = closed toll booth. Tokens refill at a steady rate.

```
  Bucket (capacity 60 tokens, refill 1 token/sec):
  ┌────────────────────────────────────────────┐
  │ ●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●●● │  ← 60 tokens
  └────────────────────────────────────────────┘
  Each call takes 1 token. Refill adds 1 token per second.
  If Mei calls 61 times in 1 second: 60 succeed, 1 gets 429.
  If Mei calls 1 time: 59 tokens left, refills to 60 in 1s.
```

In [3]:
# Safety gear #2 — TokenBucketRateLimiter
@dataclass
class TokenBucketRateLimiter:
    """Caps the number of calls per user per time window.

    The token-bucket algorithm: a bucket holds N tokens; each call
    consumes 1 token; tokens refill at rate R per second. When the
    bucket is empty, calls are rejected with a 429.
    """
    capacity: int = 60           # max tokens in the bucket
    refill_rate: float = 1.0     # tokens added per second
    tokens: float = field(default=None)
    last_refill: float = field(default=None)
    user_buckets: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.tokens is None: self.tokens = float(self.capacity)
        if self.last_refill is None: self.last_refill = time.time()

    def _refill(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_refill = now

    def allow(self, user: str = "default") -> bool:
        # Per-user bucket
        if user not in self.user_buckets:
            self.user_buckets[user] = (float(self.capacity), time.time())
        tokens, last = self.user_buckets[user]
        elapsed = time.time() - last
        tokens = min(self.capacity, tokens + elapsed * self.refill_rate)
        if tokens >= 1.0:
            self.user_buckets[user] = (tokens - 1.0, time.time())
            return True
        self.user_buckets[user] = (tokens, time.time())
        return False

# Demo
limiter = TokenBucketRateLimiter(capacity=5, refill_rate=1.0)
print("Limiter: 5 tokens, 1 token/sec refill. Simulating 8 rapid calls:")
for i in range(8):
    allowed = limiter.allow(user="mei")
    print(f"  Call {i+1}: {'✅ allowed' if allowed else '❌ 429 (rate-limited)'}")

print("\n✅ Rate limiter caps Mei at 5 calls per burst.")

Limiter: 5 tokens, 1 token/sec refill. Simulating 8 rapid calls:
  Call 1: ✅ allowed
  Call 2: ✅ allowed
  Call 3: ✅ allowed
  Call 4: ✅ allowed
  Call 5: ✅ allowed
  Call 6: ❌ 429 (rate-limited)
  Call 7: ❌ 429 (rate-limited)
  Call 8: ❌ 429 (rate-limited)

✅ Rate limiter caps Mei at 5 calls per burst.


### Safety gear #3 — The Redactor (PII stripper)

**What it does:** strips emails, phone numbers, and shipment IDs from log lines BEFORE they're written.

**Why it matters:** Singapore's PDPA (Personal Data Protection Act) and Vietnam's Decree 13/2023 both have teeth. If customer PII lands in logs, the company can be fined. The redactor makes the logs safe to grep, share with a vendor, or archive.

**The patterns:**
```
  Email:       [a-z]+@[a-z]+\.\w+     →  [EMAIL]
  Phone:       +\d{8,15}              →  [PHONE]
  Shipment:    PF-\d{4}              →  [SHIPMENT_ID]
  Card:        \d{4}-\d{4}-\d{4}-\d{4} →  [CARD]
```

In [4]:
# Safety gear #3 — Redactor
class Redactor:
    """Strips PII from log lines.

    Pattern-based, runs in O(n) on the input. Should be called BEFORE
    writing to any log file or sending to any external system.
    """
    PATTERNS = {
        "email":      re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
        "phone":      re.compile(r"\+?\d[\d\s\-]{7,}\d"),
        "shipment":   re.compile(r"PF-\d{4}"),
        "card":       re.compile(r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b"),
    }

    def redact(self, text: str) -> tuple[str, dict]:
        """Returns (redacted_text, counts_of_each_pattern_replaced)."""
        counts = {}
        for name, pattern in self.PATTERNS.items():
            text, n = pattern.subn(f"[{name.upper()}_REDACTED]", text)
            if n > 0:
                counts[name] = n
        return text, counts

# Demo
redactor = Redactor()
log_lines = [
    "Email from linh.nguyen@example.com about PF-1003 received at 2026-10-09",
    "Phone: +65 9123 4567, asked about customs",
    "Card 4242-4242-4242-4242 was charged",
    "Nothing sensitive in this line",
]
print("Redacting 4 log lines:")
for line in log_lines:
    redacted, counts = redactor.redact(line)
    count_str = ", ".join(f"{k}={v}" for k, v in counts.items()) or "(none)"
    print(f"  {count_str:40s} → {redacted}")

print("\n✅ Redactor strips PII. Logs are now safe.")

Redacting 4 log lines:
  email=1, phone=1, shipment=1             → Email from [EMAIL_REDACTED] about [SHIPMENT_REDACTED] received at [PHONE_REDACTED]
  phone=1                                  → Phone: [PHONE_REDACTED], asked about customs
  phone=1                                  → Card [PHONE_REDACTED] was charged
  (none)                                   → Nothing sensitive in this line

✅ Redactor strips PII. Logs are now safe.


### Safety gear #4 — The TTL Cache

**What it does:** if Mei submits the same email 5 times in 5 minutes, the LLM is called ONCE. The other 4 calls return the cached response.

**Why it matters:** Mei often re-renders to compare two drafts. Without a cache, every re-render costs $0.001. With a cache, only the first one costs.

**The cache key:** SHA-256 of (email text + shipment ID + prompt version). The prompt version is a hash of the system prompt. If we change the system prompt, the cache key changes, and the old cached entries are effectively invalidated (they'll expire on their own in 5 min).

```
  key = sha256(email + shipment_id + prompt_version)
  value = (draft_text, timestamp)
  TTL = 5 minutes (300 seconds)
```

In [5]:
# Safety gear #4 — TTLCache
class TTLCache:
    """Time-to-live cache. Entries expire after `ttl_s` seconds.

    Used for caching LLM responses by (email, shipment_id, prompt_version).
    Standard pattern: a 5-minute TTL means Mei's re-renders are free,
    but the cache doesn't grow unbounded.
    """
    def __init__(self, ttl_s: float = 300.0, max_size: int = 1000):
        self.ttl_s = ttl_s
        self.max_size = max_size
        self.store: dict[str, tuple[Any, float]] = {}

    def get(self, key: str) -> Optional[Any]:
        if key not in self.store:
            return None
        value, ts = self.store[key]
        if time.time() - ts > self.ttl_s:
            del self.store[key]
            return None
        return value

    def set(self, key: str, value: Any):
        if len(self.store) >= self.max_size:
            # Evict the oldest entry
            oldest = min(self.store, key=lambda k: self.store[k][1])
            del self.store[oldest]
        self.store[key] = (value, time.time())

    def stats(self) -> dict:
        return {"size": len(self.store), "ttl_s": self.ttl_s, "max_size": self.max_size}

cache = TTLCache(ttl_s=5.0)
cache.set("PF-1003-v1", "Hi Linh, your parcel is in transit.")
print(f"Cache stats: {cache.stats()}")
print(f"Get: {cache.get('PF-1003-v1')!r}")
print(f"Get (missing): {cache.get('PF-9999')!r}")

print("\n✅ TTLCache is set. Re-renders are free for 5 minutes.")

Cache stats: {'size': 1, 'ttl_s': 5.0, 'max_size': 1000}
Get: 'Hi Linh, your parcel is in transit.'
Get (missing): None

✅ TTLCache is set. Re-renders are free for 5 minutes.


### Safety gear #5 — Tiered Fallback (3 levels of backup)

**What it does:** when the LLM call fails, fall back in order:

```
  Level 1: Live LLM call (OpenAI, Anthropic, etc.)
                │
                │  fails
                ▼
  Level 2: Cached response (from TTLCache or a warmup call earlier)
                │
                │  miss
                ▼
  Level 3: Static template (a pre-written apology + status placeholder)
                │
                ▼
  Return whatever the highest-available level returned.
```

**Why it matters:** Mei always gets SOMETHING back. Even when OpenAI is down for an hour, the drafter doesn't hang. Mei sees the cached answer (good) or a template (acceptable). She keeps working.

**The analogy:** a flashlight with 3 batteries. If the main one dies, the backup kicks in. If the backup dies, the emergency battery. Mei is never in the dark.

In [6]:
# Safety gear #5 — Tiered fallback
def make_tiered_fallback(
    primary: Callable[[], str],
    cache_lookup: Callable[[], Optional[str]],
    template: str,
) -> Callable[[], str]:
    """3-level fallback: primary → cache → template.

    Returns a function that tries primary, then cache, then template.
    The function NEVER raises — it always returns SOMETHING.
    """
    def call() -> str:
        # Level 1: live LLM
        try:
            return f"[LIVE] {primary()}"
        except Exception as e:
            pass
        # Level 2: cache
        cached = cache_lookup()
        if cached:
            return f"[CACHED] {cached}"
        # Level 3: template
        return f"[TEMPLATE] {template}"
    return call

# Demo: live works, live fails, cache works, cache miss
import random
def live_llm() -> str:
    if random.random() < 0.5:
        raise ConnectionError("OpenAI is down")
    return "Fresh LLM response"

def cache_lookup() -> Optional[str]:
    return "Cached response from 3 minutes ago"

fallback = make_tiered_fallback(
    primary=live_llm,
    cache_lookup=cache_lookup,
    template="We're experiencing high demand. Please try again in a few minutes.",
)
print("Running fallback 6 times (each call has 50% chance of LLM failure):")
for i in range(6):
    print(f"  Call {i+1}: {fallback()}")

print("\n✅ Fallback always returns SOMETHING. Mei is never stuck.")

Running fallback 6 times (each call has 50% chance of LLM failure):
  Call 1: [LIVE] Fresh LLM response
  Call 2: [LIVE] Fresh LLM response
  Call 3: [CACHED] Cached response from 3 minutes ago
  Call 4: [LIVE] Fresh LLM response
  Call 5: [LIVE] Fresh LLM response
  Call 6: [CACHED] Cached response from 3 minutes ago

✅ Fallback always returns SOMETHING. Mei is never stuck.


## 🏛️ FDE Lens — the industry pattern is the same

Every production AI service in 2025 has the same 5 safety gears. Different names, same shape:

```
  This course        Industry standard (2025)
  ──────────         ─────────────────────────
  CircuitBreaker     resilience4j, Polly, Hystrix (Netflix), Istio
  RateLimiter        Token bucket in Redis, Cloudflare rate-limit, NGINX limit_req
  Redactor           Microsoft Presidio, AWS Comprehend PII, private AI redaction
  TTLCache           Redis, Memcached, Caffeine (Java), functools.lru_cache
  TieredFallback     Fallback chains in LangChain, Spring Retry, Polly
  /metrics           Prometheus + Grafana, Datadog, OpenTelemetry
  /feedback          LangSmith feedback, RAGAS feedback, custom dashboards
```

**The pattern across the industry (2025):**
1. **Observability first.** Every service exposes `/metrics` (Prometheus format). Dashboards in Grafana.
2. **Resilience by default.** Circuit breakers, retries with backoff, fallback chains. resilience4j (Java) and Polly (.NET) are the standards; the Python world uses `pybreaker` and `tenacity`.
3. **Rate limit at every layer.** Edge (NGINX/Cloudflare), app (token bucket), and downstream (OpenAI's own limits).
4. **PII redaction at the log layer.** Tools like Microsoft Presidio (open source) and AWS Comprehend PII (managed) handle this. The redactor in this notebook is a 30-line version of Presidio's idea.
5. **Streaming as the default.** `/draft/stream` (SSE) is now standard. WebSockets for bidirectional.
6. **Feedback as a first-class endpoint.** `/feedback` is not optional. Without it, you can't measure drift.

**The benefit of the standard:** when you learn the 5 safety gears in this notebook, you can read ANY production AI service in the wild. They're all the same 5 gears with different brand names.

## 🧪 Test the hardened service

Now we put it all together: the 5-piece drafter, wrapped in the 5 safety gears, behind a FastAPI service. The 13 pytest cases in `service/tests/test_app.py` test this exact thing.

In [7]:
# Wire it all up
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI(title="PF Drafter (Phase 3)")
cb = CircuitBreaker(failure_threshold=3, reset_timeout_s=1.0)
limiter = TokenBucketRateLimiter(capacity=10, refill_rate=2.0)
redactor = Redactor()
cache = TTLCache(ttl_s=60.0)
metrics = {"drafts_total": 0, "feedback_total": {"up": 0, "down": 0}}

PHASE1 = Path.cwd()
while not (PHASE1 / "shared" / "shipments.json").exists() and PHASE1 != PHASE1.parent:
    PHASE1 = PHASE1.parent
if (PHASE1 / "shared" / "shipments.json").exists():
    raw = json.loads((PHASE1 / "shared" / "shipments.json").read_text())
    shipments_db = {s["id"]: s for s in raw}
else:
    shipments_db = {"PF-1003": {"id": "PF-1003", "status": "in_customs", "customer_name": "Linh Nguyen"}}

class DraftRequest(BaseModel):
    email: str = Field(..., min_length=3)
    shipment_id: Optional[str] = None
    user: str = Field("mei")

class FeedbackRequest(BaseModel):
    draft_id: str
    rating: str = Field(..., pattern=r"^(up|down)$")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/circuit/state")
def circuit_state():
    return cb.status()

@app.post("/draft")
def draft(req: DraftRequest):
    # Rate limit
    if not limiter.allow(user=req.user):
        return {"error": "rate_limited"}, 429
    # Extract
    sid = req.shipment_id or (re.search(r"PF-\d{4}", req.email).group(0) if re.search(r"PF-\d{4}", req.email) else None)
    if not sid:
        return {"error": "no_shipment_id"}, 400
    # Cache lookup
    cache_key = f"{sid}::{hashlib.md5(req.email.encode()).hexdigest()[:8]}"
    cached = cache.get(cache_key)
    if cached:
        return {"draft": cached, "cached": True}
    # Lookup
    shipment = shipments_db.get(sid)
    if not shipment:
        return {"error": "not_found"}, 404
    # Draft (mock, but going through the breaker)
    result = cb.call(lambda: f"Hi {shipment['customer_name'].split()[0]}, your {sid} is {shipment['status'].replace('_', ' ')}.")
    cache.set(cache_key, result)
    metrics["drafts_total"] += 1
    # Redact the original email before logging
    redacted_email, _ = redactor.redact(req.email)
    return {"draft": result, "cached": False, "redacted_email": redacted_email}

@app.post("/feedback")
def feedback(req: FeedbackRequest):
    metrics["feedback_total"][req.rating] += 1
    return {"ok": True}

@app.get("/metrics")
def get_metrics():
    return metrics

client = TestClient(app)

# Run a small end-to-end suite
tests = []
r = client.get("/health"); tests.append(("/health", r.status_code == 200))
r = client.post("/draft", json={"email": "PF-1003?", "user": "mei"}); tests.append(("/draft hit", r.status_code == 200))
r = client.post("/draft", json={"email": "PF-1003?", "user": "mei"}); tests.append(("/draft cached", r.json().get("cached") is True))
r = client.get("/circuit/state"); tests.append(("/circuit/state", "state" in r.json()))
r = client.post("/feedback", json={"draft_id": "x", "rating": "up"}); tests.append(("/feedback", r.status_code == 200))
r = client.get("/metrics"); tests.append(("/metrics", r.json()["feedback_total"]["up"] == 1))

for name, passed in tests:
    mark = "✅" if passed else "❌"
    print(f"  {mark}  {name}")
passed = sum(1 for _, p in tests if p)
print(f"\n{passed}/{len(tests)} passed")

  ✅  /health
  ✅  /draft hit
  ✅  /draft cached
  ✅  /circuit/state
  ✅  /feedback
  ✅  /metrics

6/6 passed


## 🌙 Reflect

**Write down 3 things before you move on:**

1. The 5 safety gears and what each one protects against:
   - CircuitBreaker → OpenAI goes down
   - RateLimiter → Mei accidentally spends $50
   - Redactor → PII leaks to logs (PDPA violation)
   - TTLCache → Mei re-renders 5x
   - TieredFallback → multiple failures stacked

2. The 3 circuit-breaker states (CLOSED, OPEN, HALF-OPEN) and what triggers each transition.

3. The 4 new endpoints Phase 3 adds on top of Phase 2: `/draft/stream`, `/feedback`, `/metrics`, `/circuit/state`.

**Then answer the FDE Lens question:** if Mei reported that the drafter is "slow sometimes," which safety gear would you check first? (Hint: the rate limiter's logs would tell you if she's hitting the cap.)

## 🎓 What's next

Phase 3's drafter is production-grade. The 5 safety gears are in place. But:

- Mei still can't add new tools (refund, translate, escalate) without an engineer
- Complex multi-shipment cases still need Mei to click 5 buttons
- The bill will grow 10× if PacificFreight signs up 5 more customer teams
- We have one customer (PF); we haven't proven the FDE pattern transfers to a different customer

**Phase 4 fixes all of these.** The drafter becomes a platform with MCP tools, a multi-agent orchestrator, a distilled SLM that costs 5% as much, and a fresh engagement (the AI Data Analyst) to prove the pattern transfers. Plus 5 case studies, a portfolio, and a capstone presentation.

Go to [Phase 4 — Capstone](../../phase-4-capstone/) to see the drafter become a rocket ship.